# 1

In [1]:
import numpy as np
import pandas as pd

from fredapi import Fred
from dstapi import DstApi

In [2]:
fred = Fred(api_key="d31571eb7041dd22acc07bd947c0c72e")

#### 1.1

In [3]:
#1.1.1
#parallel requests to reduce FRED load time
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def download_state_data(state):
    for attempt in range(4):
        try:
            rgsp = fred.get_series(f"{state}RGSP")
            pop = fred.get_series(f"{state}POP")
            return state, rgsp, pop
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)

rgsp_data = {}
pop_data = {}

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(download_state_data, state)
        for state in STATES
    ]

    for future in as_completed(futures):
        state, rgsp, pop = future.result()
        rgsp_data[state] = rgsp
        pop_data[state] = pop

rgsp = pd.DataFrame(rgsp_data)
pop = pd.DataFrame(pop_data)

rgsp.index = rgsp.index.year
pop.index = pop.index.year

rgsp.index.name = "year"
pop.index.name = "year"

NameError: name 'STATES' is not defined

In [ ]:
# 1.1.1

from states import STATES

rgsp_data = {}
pop_data = {}
for s in STATES:
    rgsp_data[s] = fred.get_series(f"{s}RGSP")
    pop_data[s] =fred.get_series(f"{s}POP")

In [ ]:
display(fred.get_series_info("ALRGSP")[["title","frequency","units"]])
display(fred.get_series_info("ALPOP")[["title","frequency","units"]])

In [ ]:
rgsp = pd.DataFrame(rgsp_data)
pop = pd.DataFrame(pop_data)

rgsp.index =rgsp.index.year
rgsp = rgsp.rename_axis("year")
pop.index =pop.index.year
pop = pop.rename_axis("year")

display(rgsp.head(3))
display(rgsp.tail(3))

display(pop.head(3))
display(pop.tail(3))

In [ ]:
# 1.1.2

common_years = rgsp.index.intersection(pop.index)

rgsp = rgsp.loc[common_years]
pop = pop.loc[common_years]

valid_states =rgsp.dropna(axis=1).columns.intersection(pop.dropna(axis=1).columns)

rgsp = rgsp[valid_states]
pop = pop[valid_states]

print("Number of years:", len(rgsp.index))
print("Number of states:", len(valid_states))

In [ ]:
# 1.1.3 Real GDP per person

y = (rgsp*1000000)/(pop*1000)

display(y)


In [ ]:
# 1.1.4


t_first , t_last = y.index.min() , y.index.max()

for t in (t_first, t_last):
    row = y.loc[t]
    print(f"\nYear {t}:")
    print(f"  The state with the lowest real GDP per person is {row.idxmin()} with a real GPD of ${row.min():,.2f} US dollar per person")
    print(f"  The state with the highest real GDP per person is {row.idxmax()} with a real GPD of ${row.max():,.2f} US dollar per person")
    print(f"  The ratio between the states with the highest and lowest real GDP per person: {row.max()/row.min():.2f}")
    print(f"  The mean across the states in {t}: {row.mean():,.0f}")

In [ ]:
# 1.1.5 Plotting

import matplotlib.pyplot as plt

states_to_plot = ["NY", "SC", "MS", "NJ", "KS"]

fig = plt.figure(figsize=(12,5))
ax1 = fig.add_subplot(1,2,1)
ax2 = fig.add_subplot(1,2,2)

avg = y.mean(axis=1)
y[states_to_plot].plot(ax=ax1,linestyle="--" )
avg.plot(ax=ax1, color="black", label="Average")

ax1.set_title("Real GDP per person for five selected states")
ax1.set_xlabel("Year")
ax1.set_ylabel("US dollars per person (2017 dollars)")
ax1.legend(ncol=2, fontsize=10)

y_divided=y.div(avg, axis=0)
y_divided.plot(ax=ax2, alpha=0.5,linestyle="--")
ax2.axhline(1, color="black")

ax2.set_title("Real GDP per person relative to state average")
ax2.set_xlabel("Year")
ax2.set_ylabel("Ratio to average")
ax2.get_legend().remove()

fig.tight_layout()

#### 1.2

In [ ]:
# 1.2.1 Standard deviation

fig = plt.figure()
ax = fig.add_subplot(1,1,1)

std_log = np.log(y).std(axis=1)

ax.set_title("The standard deviation of log real GDP per person across states",pad=20)
ax.set_xlabel("year")
ax.set_ylabel("std. dev. of $\log y_{i,t}$ across states")

std_log.plot()

In [ ]:
# 1.2.2

t_first , t_last = std_log.index[0] , std_log.index[-1]
print(f"First value {t_first}: {std_log.iloc[0]:.4f}")
print(f"Last value {t_last}: {std_log.iloc[-1]:.4f}")
print(f"The lowest is {std_log.idxmin()}: {std_log.min():.4f}")
print(f"The highest is {std_log.idxmax()}: {std_log.max():.4f}")

##### 1.2.3

The standard deviation, or variability, of log real GDP per person in the US shows a large dip beginning in 2001 that reached a minimum in 2005 and increased rapidly until about 2009, when the financial crisis was in effect. It began to fall slowly again afterwards, but not as rapidly.

#### 1.3

In [ ]:
# 1.3.1

# Defining the variables
t_first = y.index.min()
t_last = y.index.max()
log_y = np.log(y)
T = t_last - t_first


# Setting op the function for the anual growth rate between t_first and t_last
g_i = (log_y.loc[t_last] - log_y.loc[t_first]) / T

# setting op the stating point for each state
log_y_start = log_y.loc[t_first]

# Making the scatterplot
fig = plt.figure(figsize = (10,7))
ax=fig.add_subplot(1,1,1)
ax.scatter(log_y_start,g_i)

ax.set_xlabel(r"$\log(y_{i,t_{first}})$")
ax.set_ylabel(r"$g_i$")
ax.set_title("The growth rate compared to real GDP per person in 1997")


# 1.3.2

# Adding the linear regression and reporting a, b and the correlation
b, a = np.polyfit(log_y_start,g_i,1)
x_line = np.linspace(log_y_start.min(),log_y_start.max(),100)
y_line = a + b * x_line
ax.plot(x_line, y_line, color="black", label = f"Regression line = {b:.4f}")
ax.legend()
print(f"a = {a:.4f}")
print(f"b = {b:.4f}")
corr = np.corrcoef(log_y_start, g_i)[0,1]
print(f"corr(g_i,log y_first) = {corr:.4f}")

# 1.3.3 Adding the five states with the highest and lowest g_i

g_i_sorted = g_i.sort_values(ascending=False)
five_highest = g_i_sorted.head(5).index
five_lowest =g_i_sorted.tail(5).index
for state in five_highest:
    ax.text(log_y_start[state],g_i_sorted[state],state,fontsize=10)
for state in five_lowest:
    ax.text(log_y_start[state],g_i_sorted[state],state,fontsize=10)
plt.show()

# 1.3.4
# Reporting lambda and h
_lambda_ = -np.log(1+b*T) / T
h = np.log(2)/_lambda_

print(f"Lambda = {_lambda_:.4f} per year")
print(f"h = {h:.1f}")



##### 1.3.5

There is a high variability from the line of best fit on the scatterplot, though there is a clear overall downwards trend. The five states with a lower GDP per capital in t_first are more poorly represented by the line of best fit; their growth rates g_i were substantially higher than the other values plotted.

#### 1.4

In [ ]:
from states import REGION

# 1.4.1 
# Using groupby to include the Regions

y_reset = y.reset_index()
y_long=y_reset.melt(id_vars= "year", var_name= "state", value_name="y")

region = pd.DataFrame(REGION.items(), columns = ["state","region"])
y_long = y_long.merge(region, on="state", how="left", validate="m:1")

print(y_long.head(2))
print(y_long.tail(2))

In [ ]:
# Finding the average for each region in each given year
region_average = y_long.groupby(["year","region"])["y"].mean()

# Finding the the average across all states
average_across_all_states = y_long.groupby("year")["y"].mean()


# Finding the ratio of each region
region_ratio = (region_average/average_across_all_states)

print(region_ratio)

# Finding number of states per region

count_states = region.groupby("region")["state"].count()
print(count_states)

In [ ]:
# 1.4.2 Plotting the four series

fig = plt.figure()

regions = ["Midwest","Northeast","South","West"]
for i, r in enumerate(regions):
    ax = fig.add_subplot(2,2,i+1)
    ax.plot(region_ratio.index,region_ratio[r])

    ax.set_title(r)
    ax.set_xlabel("year")
    ax.set_ylabel("State ratio to the national average")

fig.tight_layout()
plt.show()

# 2

## 3 A portfolio with a risky and safe asset

##### 3.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.grid':True,
                     'grid.color':'black',
                     'grid.alpha':'0.25',
                     'font.size': '12',
                     'grid.linestyle':'--'})

from PortfolioModel import PortfolioModelClass

# 3.1.1 draw returns, method from given PortfolioModelClass
model = PortfolioModelClass()
R = model.draw_returns() #computes Rt
print(f'R.shape = {R.shape}') #reports shape of R to verify, instead of printing the entire array

# 3.2.2 compare to the calibration
print(f'mean of log(R)  = {np.log(R).mean():.4f}  (mu = {model.par.mu})')
print(f'std of log(R)   = {np.log(R).std():.4f}  (sigma = {model.par.sigma})')
print(f'mean of R       = {R.mean():.4f}  (exp(mu+0.5*sigma^2) = {np.exp(model.par.mu+0.5*model.par.sigma**2):.4f})')

# 3.1.3 hist and plot
fig, axes = plt.subplots(1,2, figsize=(12,4.5))

axes[0].hist(R.flatten(), bins=100, color='blue')
axes[0].set_title(' $R_t$')
axes[0].set_xlabel('$R_t$')
axes[0].set_ylabel('frequency')
axes[0].set_xlim(0, 2.25)

Rf = np.exp(model.par.r)
risky_path = np.cumprod(R[:20,:], axis=1)
safe_path = Rf**np.arange(1, model.par.T+1)
for i in range(20):
    axes[1].plot(risky_path[i,:], color='blue', alpha=0.4, linewidth=1)
axes[1].plot(safe_path, color='black', linewidth=2, label='safe asset')
axes[1].set_yscale('log')
axes[1].set_title('Value of 1 invested: risky (blue) vs. safe (black)')
axes[1].set_xlabel('period')
axes[1].legend()

fig.tight_layout()

#### 3.2

In [ ]:
# 3.2.1
def run_analysis(): 
    
    model0 = PortfolioModelClass(Delta=0.0, tau=0.0)
    model1 = PortfolioModelClass(Delta=1.0, tau=0.0)
    returns = model0.draw_returns()
    model0.simulate(R=returns)
    model1.simulate(R=returns)
    return model0, model1

# 3.2.2 summary table
def print_summaries(models): 
    for name, model in models:
        summary = model.summary()
        print(name)
        print(f"  Average trades:             {summary.avg_trades:.4f}")
        print(f"  Average distance to target: {summary.avg_dist:.4f}")
        print(f"  Mean of terminal wealth:    {summary.mean_WT:.4f}")
        print(f"  Median of terminal wealth:  {summary.median_WT:.4f}")
        print(f"  10th percentile of wealth:  {summary.p10_WT:.4f}")
        print(f"  Expected utility:           {summary.EU:.4f}")

#3.2.3 plots
def make_plots(model0, model1): #plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    ax0 = axes[0]
    ax0.hist(model0.sim.W[:, -1], bins=100, range=(0, 20), density=True,
             alpha=0.5, label=r"Trade Every Period ($\Delta = 0.0$)",
             color="blue")
    ax0.hist(model1.sim.W[:, -1], bins=100, range=(0, 20), density=True,
             alpha=0.5, label=r"Never Trade ($\Delta = 1.0$)",
             color="orange")
    ax0.set_xlabel("Terminal wealth $W_T$")
    ax0.set_ylabel("Probability density")
    ax0.set_title("Terminal wealth $W_T$")
    ax0.legend(fontsize=14)

    ax1 = axes[1]
    time_grid = np.arange(model0.par.T + 1)
    for model, label, color in [
        (model0, r"$\Delta = 0.0$", "blue"),
        (model1, r"$\Delta = 1.0$", "orange"),
    ]:
        mean_theta = model.sim.theta.mean(axis=0)
        p10_theta = np.percentile(model.sim.theta, 10, axis=0)
        p90_theta = np.percentile(model.sim.theta, 90, axis=0)
        ax1.plot(time_grid, mean_theta, color=color, linewidth=2, label=label)
        ax1.fill_between(time_grid, p10_theta, p90_theta, color=color, alpha=0.15)


    ax1.set_xlabel("Period $t$")
    ax1.set_ylabel("$\\theta_t$")
    ax1.set_title("$\\theta_t$ over time")
    ax1.legend(fontsize=14)
    fig.tight_layout()



models = run_analysis()

print_summaries([
    ("Rule 1: Delta = 0.0 (Trade every period)", models[0]),
    ("Rule 2: Delta = 1.0 (Never trade)", models[1]),
])

make_plots(*models)
plt.show()


##### 3.2.4

With delta=1 the risky asset has a positive expected return, the average theta_t climbs from 0.50 to about 0.78 by period 40, and its 10th–90th percentile band widens. Delta=1, or never-trading, gives a higher mean terminal wealth W_t (8.98 vs 5.04 for delta=0) from more risk exposure but a lower expected utility (−0.0767 vs −0.0678 for delta=0). Trading every period maximizes expected utility despite returning lower average terminal wealth.

### 3.3

In [ ]:
import pandas as pd

# 3.3.1 returns for listed bandwidths with tau=0.01; draw_returns from portfoliomodelclass
base = PortfolioModelClass(tau=0.01)
R = base.draw_returns()
deltas = [0, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30, 1]

rows = []
for d in deltas:
    m = PortfolioModelClass(Delta=d, tau=0.01)
    m.simulate(R=R)
    s = m.summary()
    rows.append({'Delta':d, 'avg_trades':s.avg_trades, 'avg_dist':s.avg_dist,
                 'mean_WT':s.mean_WT, 'median_WT':s.median_WT, 'p10_WT':s.p10_WT, 'EU':s.EU})

table = pd.DataFrame(rows).set_index('Delta').round(4)
display(table)

# 3.3.2 plots
fig, axes = plt.subplots(1,2, figsize=(12,4.5)) 

ax1 = axes[0]
ax1.plot(table.index, table['avg_trades'], 'o-', color='blue')
ax1.set_xlabel('$\Delta$'); ax1.set_ylabel('avg. number of trades', color='blue')
ax2 = ax1.twinx()
ax2.plot(table.index, table['avg_dist'], 's-', color='red')
ax2.set_ylabel('avg. distance to target', color='red')
ax1.set_title('Trading activity vs. band width')
ax1.tick_params(axis='y', labelsize=8) #tick labels resize
ax2.tick_params(axis='y', labelsize=8)

best_delta = table['EU'].idxmax() #define max Delta
axes[1].plot(table.index, table['EU'], 'o-', color='green')
axes[1].set_xlabel('$\Delta$'); axes[1].set_ylabel('$E[u(W_T)]$')
axes[1].set_title('Expected utility vs. band width')
axes[1].tick_params(axis='y', labelsize=8) 

fig.tight_layout()

# 3.3.3 comparison 
print(f"best Delta = {best_delta}, EU = {table.loc[best_delta,'EU']:.4f}") #locate values corresponding to max Delta
print(f"improvement over Delta=0: {table.loc[best_delta,'EU'] - table.loc[0,'EU']:+.4f}")
print(f"improvement over Delta=1: {table.loc[best_delta,'EU'] - table.loc[1,'EU']:+.4f}")

##### 3.3.3
The highest expected utility is given by delta=0.075, beating delta=0 by 0.0006 and delta=1 by 0.0074.

##### 3.3.4
As delta increases from 0 to 0.075, average trades falls a lot (from 39 to 8.8) with only a small change in distance-to-target (from 0.039 to 0.048) or in expected utility. 
Ranking by mean W_T instead of expected utility sees mean wealth increases significantly with delta=1 over delta=0 as shown in the table. This results in favoring the rule delta=1 (never trade), despite that rule decreasing expected utility.